# Métricas de test por imagen

Este notebook evalúa en el **conjunto de test externo** la **mejor configuración**
de cada modelo (la de mayor F1 medio en desarrollo) y guarda las métricas
**imagen a imagen**.

- La mejor configuración de cada modelo se lee de `resultados/<modelo>/resumen.csv`.
- El modelo se reajusta con **todo el split de desarrollo**.
- Se predicen las **30 imágenes de test** y se calculan las métricas de cada una.
- Se genera un CSV por modelo con **30 filas** (una por imagen) en
  `resultados/test_por_imagen/<modelo>.csv`.
- También se guarda `resultados/test_por_imagen/todos.csv` con la unión de todos
  los modelos (30 filas por modelo).

## 1. Preparación 

In [7]:
import os
import sys
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if not (BASE / "aux.py").exists():
    candidato = BASE / "Proyecto_final"
    if (candidato / "aux.py").exists():
        BASE = candidato

os.chdir(BASE)
sys.path.insert(0, str(BASE))

import aux
import mean_shift as ms
import segmentacion_regiones as sr
from metricas import calcular_metricas

print("Configuraciones de desarrollo:", len(aux.rutas_configs(ms.RUTA_GRID)))
print("Imágenes de test:", len(os.listdir(aux.RUTA_TEST)))

Configuraciones de desarrollo: 66
Imágenes de test: 30


## 2. Evaluación por imagen de cada modelo

Cada modelo tiene una función que:

1. Localiza la carpeta de la configuración con `ruta_config`.
2. Reajusta el modelo con todo el split de desarrollo.
3. Predice cada imagen de test y calcula sus métricas.

Las funciones reutilizan el código de cada script de cada modelo, pero devuelven una fila por imagen en lugar de la
media.

In [8]:
def evaluar_test_mean_shift(fila):
    """Evalúa la mejor config de Mean Shift y devuelve una fila por imagen."""
    ruta = ms.ruta_config(fila["config"])
    escala = aux.datos_config(ruta)["escala"]

    nombres_dev, dev = aux.leer_dev_completo(ruta)
    ruta_test = aux.ruta_test(ruta)
    nombres_test, test = aux.leer_test(ruta_test)

    muestra, etiquetas, _ = aux.cargar_datos(nombres_dev, dev, [], ruta)
    validacion = aux.cargar_pilas(
        nombres_test, test, ruta_test, ruta_mascaras=aux.RUTA_TEST_MASCARAS)

    cols = [aux.CANALES.index(c) for c in aux.GRUPOS[fila["espacio"]]]
    bins = int(fila["bins"])
    asignacion, total_modos, manzana_modos = ms.ajustar(
        muestra, etiquetas, cols, bins, float(fila["sigma"]))
    tabla = ms.tabla_etiquetas(
        total_modos, manzana_modos, float(fila["frac_min"]))[asignacion]

    filas = []
    for nombre, (pila, ref) in zip(nombres_test, validacion):
        pred = ms.predecir(pila, cols, bins, tabla, escala)
        filas.append({"imagen": nombre, **calcular_metricas(ref, pred)})
    return filas


def evaluar_test_regiones(fila):
    """Evalúa la mejor config de regiones y devuelve una fila por imagen."""
    ruta = sr.ruta_config(fila["config"])
    escala = aux.datos_config(ruta)["escala"]

    nombres_dev, dev = aux.leer_dev_completo(ruta)
    ruta_test = aux.ruta_test(ruta)
    nombres_test, test = aux.leer_test(ruta_test)

    muestra, es_manzana, _ = aux.cargar_datos(nombres_dev, dev, [], ruta)
    datos_test = aux.cargar_pilas(
        nombres_test, test, ruta_test, ruta_mascaras=aux.RUTA_TEST_MASCARAS)

    modelo = sr.ajustar(aux.seleccionar(muestra, fila["espacio"]), es_manzana)
    mapas = sr.mapas_probabilidad(
        datos_test, fila["espacio"], modelo, fila["sigma"] * escala)

    filas = []
    for nombre, (p, ref) in zip(nombres_test, mapas):
        pred = sr.segmentar(p, fila["t_semilla"], fila["t_region"], escala)
        filas.append({"imagen": nombre, **calcular_metricas(ref, pred)})
    return filas

## 3. Registro de modelos

`MODELOS` asocia cada nombre con su `resumen.csv` y su función de evaluación.
Para añadir un modelo nuevo (por ejemplo `split_merge`) basta con:

1. Importar su módulo en la celda de preparación.
2. Escribir una función `evaluar_test_<modelo>(fila)` que devuelva una lista de
   diccionarios con `imagen` y las métricas.
3. Añadir una entrada a `MODELOS` con su `resumen.csv` y esa función.

In [9]:
MODELOS = {
    "mean_shift": {
        "resumen": BASE / "resultados" / "mean_shift" / "resumen.csv",
        "evaluar": evaluar_test_mean_shift,
    },
    "regiones": {
        "resumen": BASE / "resultados" / "regiones" / "resumen.csv",
        "evaluar": evaluar_test_regiones,
    },
}

## 4. Selección de la mejor configuración y guardado

Para cada modelo se toma la fila con mayor `f1` de su `resumen.csv`, se evalúa
en test y se escribe un CSV con una fila por imagen. 

In [10]:
RUTA_SALIDA = BASE / "resultados" / "test_por_imagen"
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

METRICAS = ("tp", "fp", "fn", "tn", "accuracy", "precision", "recall",
            "fpr", "specificity", "iou", "f1")


def metadatos(fila):
    """Columnas de la configuración elegida (sin métricas ni desviaciones)."""
    return {k: v for k, v in fila.items()
            if k not in METRICAS and not k.endswith("_std")}


partes = []
elegidas = []

for nombre_modelo, info in MODELOS.items():
    resumen = pd.read_csv(info["resumen"])
    mejor = resumen.sort_values("f1", ascending=False).iloc[0]
    elegidas.append({"modelo": nombre_modelo,
                     "config": mejor["config"],
                     "f1_dev": mejor["f1"]})

    filas = info["evaluar"](mejor)
    df = pd.DataFrame(filas)

    df.insert(0, "modelo", nombre_modelo)
    for posicion, (clave, valor) in enumerate(metadatos(mejor).items(), start=1):
        df.insert(posicion, clave, valor)

    ruta_csv = RUTA_SALIDA / f"{nombre_modelo}.csv"
    df.to_csv(ruta_csv, index=False)
    partes.append(df)
    print(f"{nombre_modelo}: {len(df)} filas -> {ruta_csv}")

df_todos = pd.concat(partes, ignore_index=True)
ruta_todos = RUTA_SALIDA / "todos.csv"
df_todos.to_csv(ruta_todos, index=False)
print(f"Combinado: {len(df_todos)} filas -> {ruta_todos}")

pd.DataFrame(elegidas)

mean_shift: 30 filas -> /Users/marcfax/Desktop/VPC/Proyecto_final/resultados/test_por_imagen/mean_shift.csv
regiones: 30 filas -> /Users/marcfax/Desktop/VPC/Proyecto_final/resultados/test_por_imagen/regiones.csv
Combinado: 60 filas -> /Users/marcfax/Desktop/VPC/Proyecto_final/resultados/test_por_imagen/todos.csv


,modelo,config,f1_dev
0,mean_shift,e1_gaussiano_k3_g1.5,0.578927
1,regiones,e1_mediana_k7_g0.7,0.665606


## 5. Resumen de resultados

Media de las métricas por modelo (calculada a partir de las 30 imágenes de
test) y primeras filas del CSV combinado.

In [11]:
resumen = (df_todos.groupby("modelo")[["accuracy", "precision", "recall",
                                       "specificity", "iou", "f1"]]
           .mean()
           .round(4))
resumen

,accuracy,precision,recall,specificity,iou,f1
modelo,,,,,,
mean_shift,0.9760,0.7035,0.1664,0.9994,0.1293,0.2101
regiones,0.9813,0.6704,0.5055,0.9961,0.3679,0.5206
